In [3]:
import sys
import os
import pandas as pd
import numpy as np

from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
from collections import Counter

# --------------------------------------------------
# 🔧 Configuración del entorno
# --------------------------------------------------

sys.path.append(os.path.abspath(".."))
from src.models.prepare_ml_data import prepare_ml_data

# --------------------------------------------------
# 📥 Carga y preparación de datos
# --------------------------------------------------

df = pd.read_csv("../data/processed/model_dataset.csv")

# Orden cronológico (evita data leakage)
df = df.sort_values("date")

# Split temporal
train_size = int(len(df) * 0.8)
train_df = df.iloc[:train_size]
test_df = df.iloc[train_size:]

# Features y target
X_train, y_train = prepare_ml_data(train_df)
X_test, y_test = prepare_ml_data(test_df)

# --------------------------------------------------
# ⚖️ Manejo de desbalance de clases
# --------------------------------------------------

counter = Counter(y_train)
total = sum(counter.values())

class_weights = {
    cls: total / count for cls, count in counter.items()
}

sample_weights = y_train.map(class_weights)

# --------------------------------------------------
# 🚀 Modelo: XGBoost
# --------------------------------------------------

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42
)

model.fit(X_train, y_train, sample_weight=sample_weights)

# --------------------------------------------------
# 🔮 Predicciones
# --------------------------------------------------

y_pred = model.predict(X_test)
probs = model.predict_proba(X_test)

# --------------------------------------------------
# 📊 Evaluación - Modelo base
# --------------------------------------------------

print("=== XGBOOST (BASE) ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# --------------------------------------------------
# 🔍 Análisis de probabilidades
# --------------------------------------------------

print("\n=== EJEMPLOS DE PROBABILIDADES ===")
for i in range(5):
    print(f"Probs: {probs[i]} → Pred: {y_pred[i]} → Real: {y_test.iloc[i]}")

# --------------------------------------------------
# 🎯 Ajuste inteligente: TOP-2 LOGIC (RECOMENDADO)
# --------------------------------------------------

# Estrategia:
# - Se ordenan las probabilidades
# - Si el empate está entre las 2 mejores opciones
# - Y es suficientemente cercano a la mejor → se predice empate

CLOSE_DIFF = 0.03  # qué tan cerca debe estar del top 1

y_pred_custom = []

for p in probs:
    sorted_idx = np.argsort(p)[::-1]  # índices ordenados de mayor a menor
    
    top1 = sorted_idx[0]
    top2 = sorted_idx[1]

    # Si empate está en el top 2 y es competitivo
    if 1 in [top1, top2] and abs(p[1] - p[top1]) < CLOSE_DIFF:
        y_pred_custom.append(1)
    else:
        y_pred_custom.append(top1)

# --------------------------------------------------
# 📊 Evaluación - Modelo ajustado
# --------------------------------------------------

print("\n=== XGBOOST (TOP-2 AJUSTADO) ===")
print("Accuracy:", accuracy_score(y_test, y_pred_custom))
print(classification_report(y_test, y_pred_custom))


KeyError: 'home_form_points'

In [32]:
X.isna().sum()


form_points           0
form_goals_for        0
form_goals_against    0
form_points_diff      0
goal_diff_recent      0
home_elo              0
away_elo              0
elo_diff              0
dtype: int64

In [16]:
y.value_counts()

target
0    19321
2    19320
1    11738
Name: count, dtype: int64

In [17]:
X.head(), y.head()

(   form_points  form_goals_for  form_goals_against  form_points_diff  \
 3          0.0             1.0                 2.0               0.0   
 4          0.0             1.0                 2.0               0.0   
 5          0.0             1.0                 2.0               0.0   
 6          3.0             2.0                 1.0               3.0   
 7          3.0             2.0                 1.0               3.0   
 
    goal_diff_recent     home_elo     away_elo   elo_diff  
 3              -1.0  1471.133526  1509.441486 -38.307960  
 4              -1.0  1500.000000  1500.000000   0.000000  
 5              -1.0  1500.000000  1500.000000   0.000000  
 6               1.0  1500.000000  1500.000000   0.000000  
 7               1.0  1519.424989  1500.000000  19.424989  ,
 3    0
 4    1
 5    1
 6    0
 7    0
 Name: target, dtype: int64)

In [33]:
importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance.head(10))

              feature  importance
7            elo_diff    0.426586
6            away_elo    0.117919
5            home_elo    0.114086
3    form_points_diff    0.070714
4    goal_diff_recent    0.069975
2  form_goals_against    0.069547
1      form_goals_for    0.066564
0         form_points    0.064610


Features extra (subir accuracy)
is_home (ventaja local explícita)
ranking_diff
days_rest_diff
tournament_importance